# ⚡ Notebook 02 — Event Stream Simulator
## Projeto: Otimização de Frotas Citi Bike NYC
### Integrante 1 — Streaming & Event Simulator

---

**Objetivo:** Simular um fluxo de dados em tempo real de viagens do Citi Bike, reproduzindo a "pressão logística" que o sistema sofre ao longo do dia. O simulador lê dados históricos da camada Bronze e os emite como micro-lotes (micro-batches), simulando um cenário de streaming.

**Arquitetura:**
```
┌─────────────────────┐     ┌─────────────────────┐     ┌──────────────────┐
│  Bronze/trips       │ ──► │  Event Simulator     │ ──► │  streaming/       │
│  (Parquet histórico)│     │  (Python Generator)  │     │  events/          │
└─────────────────────┘     │  - Micro-batches     │     │  (JSON micro-lotes│
                            │  - Velocidade ajust. │     │   particionados)  │
┌─────────────────────┐     │  - Enriquecimento    │     └──────────────────┘
│  GBFS Station Feed  │ ──► │    com station status│              │
│  (tempo real)       │     └─────────────────────┘              ▼
└─────────────────────┘                              ┌──────────────────┐
                                                     │  Spark Structured│
                                                     │  Streaming       │
                                                     │  (consumidor)    │
                                                     └──────────────────┘
```

**Ferramentas de Big Data:**
- **Spark Structured Streaming** — Consumidor dos micro-lotes
- **PySpark** — Processamento do stream
- **JSON micro-batch** — Formato intermediário do stream

## 0. Setup

In [1]:
!pip install pyspark pyarrow -q

In [2]:
import os
import sys
import json
import time
import shutil
import random
import logging
import threading
from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    TimestampType, IntegerType, LongType
)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('citibike_streaming')


In [3]:
# ============================================================
# CONFIGURAÇÃO
# ============================================================

PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR = PROJECT_ROOT / 'dados'
BRONZE_TRIPS = DATA_DIR / 'bronze' / 'trips'
BRONZE_STATIONS = DATA_DIR / 'bronze' / 'stations'

# Diretórios do streaming — usar filesystem Linux (evita problemas de permissão no WSL)
STREAM_DIR = Path.home() / 'citibike' / 'streaming'
STREAM_EVENTS = STREAM_DIR / 'events'
STREAM_CHECKPOINT = STREAM_DIR / 'checkpoint'
STREAM_OUTPUT = STREAM_DIR / 'output'

# Limpar diretórios de streaming (fresh start)
for d in [STREAM_EVENTS, STREAM_CHECKPOINT, STREAM_OUTPUT]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

# Parâmetros do simulador
BATCH_SIZE = 50
BATCH_INTERVAL_SEC = 2.0
TOTAL_BATCHES = 100
SIMULATION_DATE = '2024-07-15'
SPEED_FACTOR = 60

print(f"📁 Stream events dir: {STREAM_EVENTS}")
print(f"⚙️  Batch size: {BATCH_SIZE} viagens")
print(f"⏱️  Intervalo: {BATCH_INTERVAL_SEC}s")
print(f"📅  Data simulada: {SIMULATION_DATE}")
print(f"⚡ Speed factor: {SPEED_FACTOR}x")

📁 Stream events dir: /home/joao/citibike/streaming/events
⚙️  Batch size: 50 viagens
⏱️  Intervalo: 2.0s
📅  Data simulada: 2024-07-15
⚡ Speed factor: 60x


In [4]:
# Inicializar Spark
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("CitiBike_StreamSimulator")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "4")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark {spark.version} inicializado")
print(f"   Cores: {spark.sparkContext.defaultParallelism}")
print(f"   Driver memory: {spark.conf.get('spark.driver.memory')}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/29 23:44:04 WARN Utils: Your hostname, DESKTOP-K9JUL83, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/29 23:44:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 23:44:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark 4.1.1 inicializado
   Cores: 12
   Driver memory: 4g


---
## 1. 🏗️ Preparar Dados para Simulação

Carregamos um dia de dados históricos da Bronze e os ordenamos cronologicamente para reproduzir o fluxo real.

In [5]:
# ============================================================
# 1.1 Carregar dados de um dia da Bronze
# ============================================================

sim_date = pd.Timestamp(SIMULATION_DATE)

partition_path = BRONZE_TRIPS / f"year={sim_date.year}" / f"month={sim_date.month}"

if partition_path.exists() and any(partition_path.rglob("*.parquet")):
    logger.info(f"📖 Lendo dados de {partition_path}...")
    day_df = (
        spark.read.parquet(str(partition_path))
        .filter(F.to_date("started_at") == SIMULATION_DATE)
        .orderBy("started_at")
    )
    total_trips = day_df.count()
    logger.info(f"📊 Viagens no dia {SIMULATION_DATE}: {total_trips:,}")
    day_df.show(5, truncate=False)
else:
    logger.warning(f"⚠️  Partição não encontrada em {partition_path}")
    logger.info("🔄 Gerando dados sintéticos para demonstração...")
    total_trips = 0


23:44:07 [INFO] 📖 Lendo dados de /mnt/c/Users/PC Gamer/Documents/teste/dados/bronze/trips/year=2024/month=7...
23:44:10 [INFO] 📊 Viagens no dia 2024-07-15: 146,831                           
[Stage 4:===================================>                      (8 + 5) / 13]

+----------------+-------------+-----------------------+-----------------------+---------------------------+----------------+---------------------------+--------------+-----------------+------------------+-----------+------------+-------------+-----------------+--------------------------+
|ride_id         |rideable_type|started_at             |ended_at               |start_station_name         |start_station_id|end_station_name           |end_station_id|start_lat        |start_lng         |end_lat    |end_lng     |member_casual|trip_duration_sec|ingestion_timestamp       |
+----------------+-------------+-----------------------+-----------------------+---------------------------+----------------+---------------------------+--------------+-----------------+------------------+-----------+------------+-------------+-----------------+--------------------------+
|A97DF22F69121B25|classic_bike |2024-07-15 00:00:00.029|2024-07-15 00:04:44.557|Crown St & Troy Ave        |3751.05         |Easte

In [6]:
# ============================================================
# 1.2 Fallback: Gerador de dados sintéticos
# ============================================================

# Estações mais populares do Citi Bike (dados reais)
POPULAR_STATIONS = [
    {'id': '6681.01', 'name': 'Broadway & W 60 St', 'lat': 40.7691, 'lng': -73.9819, 'capacity': 55},
    {'id': '6140.06', 'name': 'Lincoln Center', 'lat': 40.7725, 'lng': -73.9832, 'capacity': 39},
    {'id': '5905.09', 'name': 'University Pl & E 14 St', 'lat': 40.7340, 'lng': -73.9920, 'capacity': 51},
    {'id': '6584.14', 'name': 'Grand Army Plaza & Central Park S', 'lat': 40.7644, 'lng': -73.9732, 'capacity': 63},
    {'id': '5788.03', 'name': 'W 21 St & 6 Ave', 'lat': 40.7416, 'lng': -73.9943, 'capacity': 43},
    {'id': '6494.05', 'name': 'E 53 St & Madison Ave', 'lat': 40.7594, 'lng': -73.9739, 'capacity': 35},
    {'id': '5491.09', 'name': 'Central Park W & W 72 St', 'lat': 40.7757, 'lng': -73.9762, 'capacity': 47},
    {'id': '5901.09', 'name': 'Bergen St & Smith St', 'lat': 40.6862, 'lng': -73.9901, 'capacity': 25},
    {'id': '5267.08', 'name': 'Fulton St & Broadway', 'lat': 40.7101, 'lng': -74.0095, 'capacity': 41},
    {'id': '5493.02', 'name': 'Pier 40 - Hudson River Park', 'lat': 40.7277, 'lng': -74.0116, 'capacity': 59},
    {'id': '6710.04', 'name': '1 Ave & E 68 St', 'lat': 40.7652, 'lng': -73.9584, 'capacity': 30},
    {'id': '6254.11', 'name': 'Broadway & E 22 St', 'lat': 40.7403, 'lng': -73.9892, 'capacity': 39},
    {'id': '5347.03', 'name': 'Clark St & Henry St', 'lat': 40.6976, 'lng': -73.9930, 'capacity': 24},
    {'id': '7654.01', 'name': 'Bedford Ave & N 7 St', 'lat': 40.7183, 'lng': -73.9572, 'capacity': 27},
    {'id': '6116.02', 'name': 'W 41 St & 8 Ave', 'lat': 40.7565, 'lng': -73.9904, 'capacity': 45},
]

def demand_profile(hour):
    """
    Retorna um multiplicador de demanda baseado na hora do dia.
    Simula os picos de manhã (commute) e tarde (volta + lazer).
    """
    profiles = {
        0: 0.1, 1: 0.05, 2: 0.03, 3: 0.02, 4: 0.03, 5: 0.08,
        6: 0.25, 7: 0.65, 8: 1.0, 9: 0.85, 10: 0.55, 11: 0.60,
        12: 0.70, 13: 0.65, 14: 0.60, 15: 0.65, 16: 0.80, 17: 1.0,
        18: 0.95, 19: 0.75, 20: 0.55, 21: 0.40, 22: 0.25, 23: 0.15
    }
    return profiles.get(hour, 0.3)

def generate_synthetic_trip(sim_time):
    """Gera uma viagem sintética realista."""
    start_station = random.choice(POPULAR_STATIONS)
    end_station = random.choice([s for s in POPULAR_STATIONS if s['id'] != start_station['id']])
    
    # Duração baseada na distância aproximada
    dist = ((start_station['lat']-end_station['lat'])**2 + 
            (start_station['lng']-end_station['lng'])**2) ** 0.5
    duration_sec = int(max(120, min(3600, dist * 100000 + random.gauss(0, 120))))
    
    bike_type = random.choices(
        ['classic_bike', 'electric_bike'],
        weights=[0.55, 0.45]
    )[0]
    
    member_type = random.choices(
        ['member', 'casual'],
        weights=[0.70, 0.30]
    )[0]
    
    end_time = sim_time + timedelta(seconds=duration_sec)
    
    return {
        'ride_id': f"SIM_{sim_time.strftime('%Y%m%d%H%M%S')}_{random.randint(1000,9999)}",
        'rideable_type': bike_type,
        'started_at': sim_time.isoformat(),
        'ended_at': end_time.isoformat(),
        'start_station_name': start_station['name'],
        'start_station_id': start_station['id'],
        'end_station_name': end_station['name'],
        'end_station_id': end_station['id'],
        'start_lat': start_station['lat'] + random.gauss(0, 0.0001),
        'start_lng': start_station['lng'] + random.gauss(0, 0.0001),
        'end_lat': end_station['lat'] + random.gauss(0, 0.0001),
        'end_lng': end_station['lng'] + random.gauss(0, 0.0001),
        'member_casual': member_type,
        'trip_duration_sec': duration_sec,
    }

print("🔧 Gerador de dados sintéticos pronto")
print(f"   Estações populares carregadas: {len(POPULAR_STATIONS)}")
print(f"   Exemplo de viagem:")
sample = generate_synthetic_trip(datetime(2024, 7, 15, 8, 30, 0))
for k, v in sample.items():
    print(f"     {k}: {v}")

🔧 Gerador de dados sintéticos pronto
   Estações populares carregadas: 15
   Exemplo de viagem:
     ride_id: SIM_20240715083000_1743
     rideable_type: classic_bike
     started_at: 2024-07-15T08:30:00
     ended_at: 2024-07-15T09:30:00
     start_station_name: Clark St & Henry St
     start_station_id: 5347.03
     end_station_name: Bedford Ave & N 7 St
     end_station_id: 7654.01
     start_lat: 40.697567596948765
     start_lng: -73.99294778923608
     end_lat: 40.718220221197
     end_lng: -73.95701761709977
     member_casual: member
     trip_duration_sec: 3600
